# K09-04 — RAID mixtures and designs grids, shard 4 of 20 (Kaggle CPU)

Runs every 20th RAID pool of the mixtures and designs grids, starting at pool 4
(`scripts/run_s5.py --shard 4/20`): 5 pools of each grid, about 1 hour. Twenty notebooks
K09-00 ... K09-19 cover both grids. Code: commit `bcacbd7` of github.com/DRuanli/dvai
(simulation code identical to `785fb1d`). Seeds depend on the cell, so the split does not
change any result.

**Settings:** *Accelerator* = **None** (CPU), *Internet* = **On**. Then *Save & Run All*.
No input dataset is needed.

**When it finishes** (`RESULT: COMPLETE`): download **`K09_04_compact.zip`** from the
Output and send it back.


In [ ]:
import os, glob, json, shutil, subprocess, sys, time, zipfile
COMMIT = "bcacbd7f0f95de17e0b2989d9c78c09f5eb29791"
SHARD = "4/20"
REPO = "/tmp/dvai"   # outside /kaggle/working, so it is not saved with the output
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/DRuanli/dvai.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
# use the package from the checkout (an editable pip install only works after a kernel restart)
sys.path.insert(0, f"{REPO}/src")
os.environ["PYTHONPATH"] = f"{REPO}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO)
import d2vi
assert d2vi.__file__.startswith(REPO), d2vi.__file__
print("d2vi", d2vi.__version__, "| commit", COMMIT[:7], "| shard", SHARD, "| CPUs", os.cpu_count())

In [ ]:
# Resume: bring back raw files of earlier runs (attached as an input dataset)
GRIDS = ["mixtures", "designs"]
OUT = "/kaggle/working"
brought = 0
for f in glob.glob("/kaggle/input/**/raw/*/raid*", recursive=True):
    grid = os.path.basename(os.path.dirname(f))
    if grid in GRIDS:
        dst = f"{REPO}/results/raw/S5/{grid}/{os.path.basename(f)}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst) or os.path.basename(f).endswith(".json"):
            shutil.copy(f, dst); brought += 1
print("files brought back from earlier runs:", brought)

In [ ]:
# Simulations: grids in order, until 10 hours have passed (a pool takes a few minutes)
START, BUDGET_MIN = time.time(), 600
for grid in GRIDS:
    left = BUDGET_MIN - (time.time() - START) / 60
    if left <= 0:
        break
    print(f"== {grid}: {left:.0f} minutes left", flush=True)
    p = subprocess.run([sys.executable, "-u", "scripts/run_s5.py", "configs/s5/raid.yaml", "--grid", grid,
                        "--jobs", str(os.cpu_count()), "--shard", SHARD, "--max-minutes", f"{left:.0f}"],
                       capture_output=True, text=True)
    lines = p.stdout.splitlines()
    print("\n".join([l for l in lines if l.startswith(("done", "empty", "time budget"))][-200:]))
    if p.returncode:
        print(p.stdout[-3000:], p.stderr[-6000:]); raise RuntimeError(f"run_s5 failed on {grid}")

In [ ]:
# Copy raw files to the output, reduce them to compact tables, zip the compact tables
import yaml, pandas as pd
sys.path.insert(0, f"{REPO}/scripts")
import run_s5
cfg = run_s5.resolve(yaml.safe_load(open("configs/s5/raid.yaml")), pd.read_parquet("data/features/raid_features.parquet"))
k, n_sh = map(int, SHARD.split("/"))
planned = {name for pos, (name, *_) in enumerate(run_s5.pool_plan(cfg)) if pos % n_sh == k}
status = {}
for grid in GRIDS:
    src = f"{REPO}/results/raw/S5/{grid}"
    if not os.path.isdir(src):
        status[grid] = 0; continue
    os.makedirs(f"{OUT}/raw/{grid}", exist_ok=True)
    for f in glob.glob(f"{src}/raid*"):
        dst = f"{OUT}/raw/{grid}/{os.path.basename(f)}"
        if not os.path.exists(dst) or f.endswith(".json"):
            shutil.copy(f, dst)
    c = subprocess.run([sys.executable, "scripts/compact_s5.py", f"{OUT}/raw/{grid}", f"{OUT}/compact/{grid}"],
                       capture_output=True, text=True)
    if c.returncode:          # report and go on: the raw files are saved either way
        print(f"compaction of {grid} failed:", c.stderr[-2000:])
    man = json.load(open(f"{src}/raid.manifest.json")) if os.path.exists(f"{src}/raid.manifest.json") else {"pools": {}}
    status[grid] = len(set(man["pools"]) & planned)
with zipfile.ZipFile(f"{OUT}/K09_04_compact.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/compact/**/*", recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUT))
print({g: f"{n}/{len(planned)} pools" for g, n in status.items()})
done = all(n == len(planned) for n in status.values())
print("K09_04_compact.zip:", round(os.path.getsize(f"{OUT}/K09_04_compact.zip") / 1e6, 1), "MB")
print("RESULT:", "COMPLETE" if done else "PARTIAL - save the Output as a dataset, add it as Input, run again")